# Module 5 · Filtering and prioritization
**Xenoi Biotech · DNA-Seq (Variant Analysis) Workshop** · Week 3–4 · Days 21–25

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anjalianjali1234/dna-seq-variant-colab/blob/main/notebooks/M5_Filtering_and_prioritization.ipynb)

**By the end of this module:** you can narrow thousands of variants to a short list, apply inheritance and case-control logic, rank candidates and confirm them visually in IGV.

**Workshop question:** can different variant callers recover the same small DNA changes in a real human sample (HG001 / NA12878), and how do we know which calls to trust?

**How to use this notebook:** run the cells from top to bottom. Read each explanation before running the code under it. Cells marked ✍️ are exercises: write your answers in your lab notebook.

## Session setup (run every time you open Colab)
Colab gives you a fresh computer each session, so this cell installs the tools and downloads the data. The first run takes about **3–5 minutes**; go through the reading below while it works.

In [ ]:
import os
%cd /content
if not os.path.exists("dna-seq-variant-colab"):
    !git clone -q https://github.com/anjalianjali1234/dna-seq-variant-colab.git
%cd /content/dna-seq-variant-colab
!bash setup/colab_setup.sh
os.environ["PATH"] = "/content/env/bin:" + os.environ["PATH"]
!samtools --version | head -1 && gatk --version 2>/dev/null | head -1

### Catch up on earlier modules
This rebuilds the outputs of Modules 1–3 (about 2–4 minutes). Skip it if you ran them in this same session.

In [ ]:
!bash setup/catchup.sh 3

> 💾 **Optional:** to keep your outputs between sessions, mount Google Drive and copy `run/` there at the end:
> `from google.colab import drive; drive.mount('/content/drive')` then `!cp -r run /content/drive/MyDrive/dna-seq-run`

In [ ]:
!python scripts/benchmark.py 30 10 > /dev/null && python scripts/annotate.py > /dev/null
import pandas as pd
ann = pd.read_csv("run/report/annotated.tsv", sep="\t")
ann["af"] = pd.to_numeric(ann.gnomad_genomes_af, errors="coerce")
print(len(ann), "variants loaded")

## Day 21 · Filtering strategies
A real exome has ~20,000–25,000 variants. A typical funnel: quality → called by ≥2 callers → rare → damaging consequence → candidate genes.

In [ ]:
steps = [("all called variants", ann),
         ("called by at least 2 callers", ann[ann[["gatk", "bcftools", "freebayes"]].sum(axis=1) >= 2])]
steps.append(("…and frequency < 5% or unknown", steps[-1][1][(steps[-1][1].af < 0.05) | steps[-1][1].af.isna()]))
damaging = ["missense_variant", "stop_gained", "frameshift_variant", "splice_donor_variant", "splice_acceptor_variant", "start_lost", "stop_lost"]
steps.append(("…and protein-altering", steps[-1][1][steps[-1][1].consequence.isin(damaging)]))
for name, df in steps: print(f"{len(df):4d}  {name}")

> ✍️ **Your turn**
>
> 1. Why does the list shrink to very few (or zero) variants here? Is an empty list a valid result?
> 2. Change the frequency cut-off to 1%. Why should "unknown frequency" be kept as a separate group, not treated as rare?

## Day 22 · Inheritance patterns (teaching scenario)
HG001 has no family data in this lab, so we use **invented** trio genotypes to practise the logic.

In [ ]:
trio = pd.DataFrame({"locus": ["A", "B", "C", "D"],
                     "child": ["0/1", "1/1", "0/1", "0/1"], "mother": ["0/0", "0/1", "0/0", "0/1"], "father": ["0/0", "0/1", "./.", "0/0"]})
def model(r):
    if r.child == "0/1" and r.mother == "0/0" and r.father == "0/0": return "de novo?"
    if r.child == "1/1" and r.mother == "0/1" and r.father == "0/1": return "autosomal recessive"
    if "./." in (r.mother, r.father): return "cannot assess: missing parent"
    return "inherited heterozygous"
trio["candidate_model"] = trio.apply(model, axis=1); trio

> ✍️ **Your turn**
>
> 1. For locus A, what extra evidence would you need before calling it de novo? (allele balance, parentage, coverage, mosaicism)
> 2. Which two loci in one gene could suggest **compound heterozygosity**, and what data (phasing) would confirm it?

## Day 23 · Case-control logic and candidate genes
A hypothetical allele appears on 4 of 20 case chromosomes and 1 of 20 control chromosomes.

In [ ]:
from scipy.stats import fisher_exact
table = [[4, 16], [1, 19]]
print("odds ratio %.2f, p = %.3f" % fisher_exact(table))

> ✍️ **Your turn**
>
> 1. Is this evidence of association? Discuss sample size, ancestry matching and coverage.
> 2. Why can gnomAD frequency not replace matched controls?
> 3. Rank the variants in `annotated.tsv` by a simple score of your choice (e.g. callers agreeing + rarity + consequence). Explain your weights.

## Day 24 · Visual review in IGV (inside Colab)
`igv-notebook` embeds the IGV genome browser in the notebook.

In [ ]:
!pip -q install igv-notebook
import igv_notebook
igv_notebook.init()
pos = int(ann[ann.benchmark == "match"].iloc[0].pos)
b = igv_notebook.Browser({"reference": {"id": "hg19_chr20_training", "name": "hg19 chr20 (training)",
                                        "fastaPath": "run/input/ref.fa", "indexPath": "run/input/ref.fa.fai"},
                          "locus": f"chr20:{pos-60}-{pos+60}"})
b.load_track({"name": "HG001 reads", "path": "run/align/HG001.bam", "indexPath": "run/align/HG001.bam.bai", "format": "bam", "type": "alignment"})
b.load_track({"name": "GATK calls", "path": "run/calls/gatk.norm.vcf.gz", "indexPath": "run/calls/gatk.norm.vcf.gz.tbi", "format": "vcf", "type": "variant"})

If the embedded browser does not load, download `run/align/HG001.bam`, its `.bai`, `ref.fa` and `ref.fa.fai` from the Colab file panel and open them in **IGV Desktop** or at igv.org/app.

> ✍️ **Your turn**
>
> Review one shared SNV and one caller disagreement or indel. For each record: ALT read support, depth, strand balance, mapping quality, soft-clipping nearby, and your decision (retain / uncertain / reject). Take a screenshot.

## Day 25 · Prioritised shortlist
**Module 5 deliverable:** a ranked table of your top 3 variants with the evidence for each, two IGV review notes, and one paragraph on what extra data (family samples, phenotype) would allow inheritance testing.